# 1.3 Applications of Systems of Linear Equations

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 1:** Systems of Linear Equations

---

### What you will learn

- How to set up a **polynomial curve-fitting** problem as a linear system (Vandermonde systems).
- How to model **network flow** problems (e.g., traffic networks) using flow conservation equations.
- The difference between **determined** and **underdetermined** applied systems.
- How to interpret **free variables** physically (adjustable flows, tunable parameters).
- How to **verify** applied solutions both algebraically and with NumPy.

### Prerequisites

- Gaussian and Gauss-Jordan elimination (Notebook 1.2).
- The `solve()` function from `linalg_core.elimination`.
- Basic familiarity with polynomials and function evaluation.

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.elimination import (
    swap_rows,
    scale_row,
    add_scaled_row,
    to_ref,
    to_rref,
    solve,
)
from linalg_core import EPSILON

import numpy as np
import matplotlib.pyplot as plt

---

## 2. Motivation --- Polynomial Interpolation

Suppose we have collected four data points from an experiment:

$$(-2,\, 15), \quad (-1,\, 4), \quad (1,\, 2), \quad (3,\, 28)$$

We want to find **the unique polynomial** of degree at most 3 that passes through
all four points:

$$p(x) = a_0 + a_1 x + a_2 x^2 + a_3 x^3$$

Substituting each data point $(x_i, y_i)$ into this equation gives one linear
equation per point. Four points, four unknowns --- a square system.

This is a **Vandermonde system**: the coefficient matrix has the special
structure where row $i$ is $[1,\; x_i,\; x_i^2,\; \ldots,\; x_i^{n-1}]$.

**Why does this work?** The key insight from first principles is:
- A polynomial of degree $n-1$ has $n$ free coefficients.
- Each data point constrains one degree of freedom.
- So $n$ distinct points uniquely determine a degree-$(n-1)$ polynomial.

The Vandermonde matrix is guaranteed to be nonsingular when all $x_i$ are
**distinct** (a fact we will prove using determinants in Chapter 3).

In [ ]:
data_points = [(-2, 15), (-1, 4), (1, 2), (3, 28)]

print("Data points:")
for x, y in data_points:
    print(f"  ({x}, {y})")

print("\nSubstituting into p(x) = a0 + a1*x + a2*x^2 + a3*x^3:")
print()
for x, y in data_points:
    print(f"  p({x:2d}) = a0 + ({x:2d})a1 + ({x**2:2d})a2 + ({x**3:3d})a3 = {y}")

---

## 3. Build --- Core Concepts

We build up two major applications in five steps:

1. Vandermonde matrix construction
2. Polynomial curve fitting via `solve()`
3. Evaluation and plotting of the fitted polynomial
4. Network flow analysis (determined system)
5. Underdetermined network analysis (parametric solutions)

### 3.1 Vandermonde Matrix Construction

The **Vandermonde matrix** for $x$-values $x_0, x_1, \ldots, x_{m-1}$ and
polynomial degree $n-1$ is the $m \times n$ matrix:

$$
V = \begin{bmatrix}
1 & x_0 & x_0^2 & \cdots & x_0^{n-1} \\
1 & x_1 & x_1^2 & \cdots & x_1^{n-1} \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
1 & x_{m-1} & x_{m-1}^2 & \cdots & x_{m-1}^{n-1}
\end{bmatrix}
$$

Each row corresponds to one data point. Column $j$ contains $x_i^j$.
The system $V\mathbf{a} = \mathbf{y}$ has unknowns
$\mathbf{a} = [a_0, a_1, \ldots, a_{n-1}]^T$.

In [ ]:
def build_vandermonde(x_values, degree):
    """Build a Vandermonde matrix for polynomial fitting.

    Parameters
    ----------
    x_values : list of float
        The x-coordinates of the data points.
    degree : int
        The degree of the polynomial. The matrix will have (degree + 1) columns.

    Returns
    -------
    Matrix
        An m x (degree+1) Vandermonde matrix where m = len(x_values).
    """
    m = len(x_values)
    n = degree + 1
    rows = []
    for x in x_values:
        row = [x ** j for j in range(n)]
        rows.append(row)
    return Matrix.from_lists(rows)


x_vals = [x for x, y in data_points]
V = build_vandermonde(x_vals, degree=3)
print("Vandermonde matrix V:")
print(V)

### 3.2 Polynomial Curve Fitting

To find the coefficients $a_0, a_1, a_2, a_3$, we solve the augmented system
$[V \mid \mathbf{y}]$:

$$
\begin{bmatrix}
1 & -2 & 4 & -8 & 15 \\
1 & -1 & 1 & -1 & 4 \\
1 &  1 & 1 &  1 & 2 \\
1 &  3 & 9 & 27 & 28
\end{bmatrix}
$$

If our four $x$-values are distinct, the Vandermonde matrix is nonsingular,
so we expect a **unique** solution.

In [ ]:
y_vals = [y for x, y in data_points]

aug_data = []
for i in range(len(data_points)):
    row = V.get_row(i) + [y_vals[i]]
    aug_data.append(row)

aug = Matrix.from_lists(aug_data)
print("Augmented matrix [V | y]:")
print(aug)

print("\nSolving with RREF...")
sol_type, coeffs = solve(aug, verbose=True)

print(f"\nSolution type: {sol_type}")
print(f"Coefficients: a0={coeffs[0]:.4f}, a1={coeffs[1]:.4f}, "
      f"a2={coeffs[2]:.4f}, a3={coeffs[3]:.4f}")
print(f"\np(x) = {coeffs[0]:.4f} + ({coeffs[1]:.4f})x + "
      f"({coeffs[2]:.4f})x^2 + ({coeffs[3]:.4f})x^3")

### 3.3 Evaluate and Plot

To confirm the polynomial passes through our data, we:
1. Define an evaluation function `eval_polynomial(coeffs, x)`.
2. Evaluate on a fine grid for a smooth curve.
3. Plot the curve and overlay the original data points.

In [ ]:
def eval_polynomial(coeffs, x):
    """Evaluate a polynomial at a given x value.

    Parameters
    ----------
    coeffs : list of float
        Coefficients [a0, a1, a2, ...] so p(x) = a0 + a1*x + a2*x^2 + ...
    x : float
        The point at which to evaluate.

    Returns
    -------
    float
        The value p(x).
    """
    result = 0.0
    for i, a in enumerate(coeffs):
        result += a * (x ** i)
    return result


print("Checking polynomial at data points:")
for x, y in data_points:
    px = eval_polynomial(coeffs, x)
    print(f"  p({x:2d}) = {px:8.4f}  (expected {y})")

x_fine = [i * 0.05 for i in range(-50, 71)]
y_fine = [eval_polynomial(coeffs, x) for x in x_fine]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_fine, y_fine, 'b-', linewidth=2, label='$p(x) = a_0 + a_1 x + a_2 x^2 + a_3 x^3$')
ax.scatter([x for x, y in data_points], [y for x, y in data_points],
           color='red', s=80, zorder=5, label='Data points')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Polynomial Interpolation Through 4 Points', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
plt.tight_layout()
plt.show()

### 3.4 Network Flow Analysis

Linear systems arise naturally in **network flow** problems. Consider a
simple traffic network with four intersections arranged in a line. External
traffic enters and exits at certain intersections, and internal road segments
carry flow between them.

**Network layout:**

```
  300 (in)       100 (out)      150 (out)
     |               ^               ^
     v               |               |
    [1] --- x1 ---> [2] --- x2 ---> [3] --- x3 ---> [4]
                                                      |
                                                      v
                                                   x4 (out)
```

**Variables:** $x_1, x_2, x_3, x_4$ represent the flow (vehicles per hour)
along the road segments connecting the intersections.

**Flow conservation principle:** At each intersection, the total flow
entering must equal the total flow leaving. This is a direct application of
conservation laws --- the same first principle that governs electrical circuits
(Kirchhoff's current law) and fluid networks.

Writing the equations:

| Intersection | Flow In | Flow Out | Equation |
|:---:|:---:|:---:|:---:|
| 1 | 300 | $x_1$ | $x_1 = 300$ |
| 2 | $x_1$ | $100 + x_2$ | $x_1 - x_2 = 100$ |
| 3 | $x_2$ | $150 + x_3$ | $x_2 - x_3 = 150$ |
| 4 | $x_3$ | $x_4$ | $x_3 - x_4 = 0$ |

**Sanity check:** Total external inflow = 300. Total external outflow =
$100 + 150 + x_4$. For the network to balance, we need $x_4 = 50$.
Let us verify this by solving the system.

In [ ]:
network_data = [
    [1,  0,  0,  0,  300],
    [1, -1,  0,  0,  100],
    [0,  1, -1,  0,  150],
    [0,  0,  1, -1,    0],
]

network_aug = Matrix.from_lists(network_data)
print("Flow conservation system [A | b]:")
print(network_aug)

print("\nSolving...")
net_type, net_result = solve(network_aug, verbose=True)

print(f"\nSolution type: {net_type}")
if net_type == "unique":
    var_names = ['x1', 'x2', 'x3', 'x4']
    for name, val in zip(var_names, net_result):
        print(f"  {name} = {val:.0f} vehicles/hour")

print("\nInterpretation:")
print(f"  [1] -> [2]: x1 = {net_result[0]:.0f}  (all 300 entering flow passes through)")
print(f"  [2] -> [3]: x2 = {net_result[1]:.0f}  (300 minus the 100 that exit at [2])")
print(f"  [3] -> [4]: x3 = {net_result[2]:.0f}   (200 minus the 150 that exit at [3])")
print(f"  [4] out:    x4 = {net_result[3]:.0f}   (the remaining 50 exit the network)")
print(f"\nTotal external inflow:  300")
print(f"Total external outflow: 100 + 150 + {net_result[3]:.0f} = "
      f"{100 + 150 + net_result[3]:.0f}  (balanced)")

### 3.5 Underdetermined Network

Real traffic networks often have **more road segments than intersections**,
leading to underdetermined systems. Let us extend the network by adding
a **bypass road** $x_5$ directly from intersection 1 to intersection 3,
allowing traffic to skip intersection 2 entirely.

```
  300 (in)       100 (out)      150 (out)
     |               ^               ^
     v               |               |
    [1] --- x1 ---> [2] --- x2 ---> [3] --- x3 ---> [4]
     |                               ^                |
     |              x5                |                v
     +--------------------------------+             x4 (out)
```

Updated flow conservation equations (5 unknowns, 4 equations):

| Intersection | Flow In | Flow Out | Equation |
|:---:|:---:|:---:|:---:|
| 1 | 300 | $x_1 + x_5$ | $x_1 + x_5 = 300$ |
| 2 | $x_1$ | $100 + x_2$ | $x_1 - x_2 = 100$ |
| 3 | $x_2 + x_5$ | $150 + x_3$ | $x_2 - x_3 + x_5 = 150$ |
| 4 | $x_3$ | $x_4$ | $x_3 - x_4 = 0$ |

With 5 unknowns and 4 equations, we expect a **one-parameter family**
of solutions. The free variable corresponds to a flow we can
**adjust freely** --- for instance, by controlling traffic light timing
on the bypass road.

In [ ]:
underdet_data = [
    [1,  0,  0,  0,  1,  300],
    [1, -1,  0,  0,  0,  100],
    [0,  1, -1,  0,  1,  150],
    [0,  0,  1, -1,  0,    0],
]

underdet_aug = Matrix.from_lists(underdet_data)
print("Underdetermined network system [A | b]:")
print(underdet_aug)

print("\nSolving...")
ud_type, ud_result = solve(underdet_aug, verbose=True)

print(f"\nSolution type: {ud_type}")
if ud_type == "infinite":
    fv = ud_result["free_vars"]
    part = ud_result["particular"]
    dirs = ud_result["directions"]

    var_names = ['x1', 'x2', 'x3', 'x4', 'x5']
    print(f"\nFree variable(s): {[var_names[j] for j in fv]}")
    print(f"\nParticular solution (setting free vars to 0):")
    for name, val in zip(var_names, part):
        print(f"  {name} = {val:.0f}")

    print(f"\nParametric form (let t = {var_names[fv[0]]}):\n")
    for i, name in enumerate(var_names):
        coeff = dirs[fv[0]][i]
        if abs(coeff) < EPSILON:
            print(f"  {name} = {part[i]:.0f}")
        else:
            sign = '+' if coeff > 0 else '-'
            print(f"  {name} = {part[i]:.0f} {sign} {abs(coeff):.0f}t")

    print("\nPhysical interpretation:")
    print("  The bypass flow x5 = t is a free parameter.")
    print("  Adjusting t (e.g., via traffic light timing on the bypass)")
    print("  diverts traffic away from the [1]->[2] and [2]->[3] segments")
    print("  while preserving conservation at every intersection.")
    print("  Segments x3 and x4 are unaffected by the bypass.")
    print("\n  For all flows to be non-negative (physical constraint):")
    print("  we need each xi >= 0.")

    t_lower = None
    t_upper = None
    print()
    for i, name in enumerate(var_names):
        coeff = dirs[fv[0]][i]
        if abs(coeff) > EPSILON:
            bound = -part[i] / coeff
            if coeff > 0:
                print(f"    {name} = {part[i]:.0f} {'+' if coeff > 0 else '-'} "
                      f"{abs(coeff):.0f}t >= 0  =>  t >= {bound:.0f}")
                if t_lower is None or bound > t_lower:
                    t_lower = bound
            else:
                print(f"    {name} = {part[i]:.0f} {'+' if coeff > 0 else '-'} "
                      f"{abs(coeff):.0f}t >= 0  =>  t <= {bound:.0f}")
                if t_upper is None or bound < t_upper:
                    t_upper = bound
        else:
            if part[i] >= -EPSILON:
                print(f"    {name} = {part[i]:.0f} >= 0  (always satisfied)")
            else:
                print(f"    {name} = {part[i]:.0f} < 0  (INFEASIBLE)")

    if t_lower is None:
        t_lower = 0
    print(f"\n  Valid range: {t_lower:.0f} <= t <= {t_upper:.0f}")
    print(f"  The bypass can carry between {max(0, t_lower):.0f} and {t_upper:.0f} "
          f"vehicles/hour.")

---

## 4. Verify --- Cross-Check with NumPy

We verify our from-scratch results in three ways:

1. **Polynomial:** Evaluate $p(x_i)$ at each data point and check it matches $y_i$.
2. **Cross-check:** Compare coefficients against `np.polyfit` (which returns
   coefficients in **reverse** order: highest degree first).
3. **Network:** Verify all flow conservation equations are satisfied.

In [ ]:
print("=" * 60)
print("VERIFICATION 1: Polynomial passes through all data points")
print("=" * 60)

all_pass = True
for x, y in data_points:
    px = eval_polynomial(coeffs, x)
    error = abs(px - y)
    status = "PASS" if error < EPSILON else "FAIL"
    if error >= EPSILON:
        all_pass = False
    print(f"  p({x:2d}) = {px:10.6f}  expected {y:6.1f}  error = {error:.2e}  [{status}]")

print(f"\n  Overall: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

In [ ]:
print("=" * 60)
print("VERIFICATION 2: Cross-check against np.polyfit")
print("=" * 60)

x_np = np.array([x for x, y in data_points], dtype=float)
y_np = np.array([y for x, y in data_points], dtype=float)

np_coeffs_reversed = np.polyfit(x_np, y_np, 3)
np_coeffs = list(reversed(np_coeffs_reversed))

print("\n  Our coefficients (a0, a1, a2, a3):")
print(f"    [{', '.join(f'{c:10.6f}' for c in coeffs)}]")
print("\n  NumPy coefficients (reversed from np.polyfit):")
print(f"    [{', '.join(f'{c:10.6f}' for c in np_coeffs)}]")

max_diff = max(abs(a - b) for a, b in zip(coeffs, np_coeffs))
print(f"\n  Max difference: {max_diff:.2e}")
print(f"  Status: {'PASS' if max_diff < 1e-6 else 'FAIL'}")

In [ ]:
print("=" * 60)
print("VERIFICATION 3: Network flow conservation")
print("=" * 60)

det_equations = [
    ("Intersection 1: x1 = 300",           lambda s: s[0], 300),
    ("Intersection 2: x1 - x2 = 100",      lambda s: s[0] - s[1], 100),
    ("Intersection 3: x2 - x3 = 150",      lambda s: s[1] - s[2], 150),
    ("Intersection 4: x3 - x4 = 0",        lambda s: s[2] - s[3], 0),
]

print("\nDetermined network (4 segments):")
all_ok = True
for desc, lhs_fn, rhs in det_equations:
    lhs = lhs_fn(net_result)
    error = abs(lhs - rhs)
    status = "PASS" if error < EPSILON else "FAIL"
    if error >= EPSILON:
        all_ok = False
    print(f"  {desc:38s}  LHS = {lhs:8.1f}  [{status}]")
print(f"  Overall: {'ALL PASSED' if all_ok else 'SOME FAILED'}")

if ud_type == "infinite":
    print("\nUnderdetermined network (5 segments, testing with t=0):")
    test_sol = list(ud_result["particular"])
    ud_equations = [
        ("Intersection 1: x1 + x5 = 300",      lambda s: s[0] + s[4], 300),
        ("Intersection 2: x1 - x2 = 100",      lambda s: s[0] - s[1], 100),
        ("Intersection 3: x2 - x3 + x5 = 150", lambda s: s[1] - s[2] + s[4], 150),
        ("Intersection 4: x3 - x4 = 0",        lambda s: s[2] - s[3], 0),
    ]
    all_ok = True
    for desc, lhs_fn, rhs in ud_equations:
        lhs = lhs_fn(test_sol)
        error = abs(lhs - rhs)
        status = "PASS" if error < EPSILON else "FAIL"
        if error >= EPSILON:
            all_ok = False
        print(f"  {desc:38s}  LHS = {lhs:8.1f}  [{status}]")
    print(f"  Overall: {'ALL PASSED' if all_ok else 'SOME FAILED'}")

    print("\n  Also testing with t=100 (midpoint of valid range):")
    t_test = 100
    test_sol_t = [part[i] + dirs[fv[0]][i] * t_test for i in range(5)]
    print(f"    Solution: {['%s=%.0f' % (var_names[i], test_sol_t[i]) for i in range(5)]}")
    all_ok = True
    for desc, lhs_fn, rhs in ud_equations:
        lhs = lhs_fn(test_sol_t)
        error = abs(lhs - rhs)
        if error >= EPSILON:
            all_ok = False
    print(f"    Conservation check: {'ALL PASSED' if all_ok else 'SOME FAILED'}")

---

## 5. Visualize

Two visualizations bring our results to life:

1. A scatter plot of data points with the smooth interpolating polynomial.
2. A network diagram showing intersections, edges, and flow values.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

x_fine = [i * 0.02 for i in range(-130, 181)]
y_fine = [eval_polynomial(coeffs, x) for x in x_fine]

ax.plot(x_fine, y_fine, color='#2563eb', linewidth=2.5, label='Interpolating polynomial')

ax.scatter([x for x, y in data_points], [y for x, y in data_points],
           color='#dc2626', s=100, zorder=5, edgecolors='white',
           linewidth=1.5, label='Data points')

for x, y in data_points:
    ax.annotate(f'({x}, {y})', (x, y), textcoords='offset points',
                xytext=(8, 10), fontsize=10, color='#374151')

coeff_str = (f"$p(x) = {coeffs[0]:.1f}"
             f" {'+' if coeffs[1] >= 0 else '-'} {abs(coeffs[1]):.1f}x"
             f" + {coeffs[2]:.1f}x^2"
             f" + {coeffs[3]:.2f}x^3$")
ax.text(0.05, 0.95, coeff_str, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', bbox=dict(boxstyle='round,pad=0.4',
        facecolor='#eff6ff', edgecolor='#93c5fd', alpha=0.9))

ax.set_xlabel('x', fontsize=13)
ax.set_ylabel('y', fontsize=13)
ax.set_title('Polynomial Curve Fitting via Vandermonde System', fontsize=14)
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(-0.5, 8.5)
ax.set_ylim(-1.5, 2.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Traffic Network --- Determined System (4 Segments)', fontsize=14, pad=15)

nodes = {1: (1, 1), 2: (3, 1), 3: (5, 1), 4: (7, 1)}

for nid, (nx, ny) in nodes.items():
    circle = plt.Circle((nx, ny), 0.3, color='#2563eb', zorder=5)
    ax.add_patch(circle)
    ax.text(nx, ny, str(nid), ha='center', va='center',
            fontsize=14, fontweight='bold', color='white', zorder=6)

internal_color = '#1e40af'
external_color = '#16a34a'

internal_edges = [
    (1, 2, net_result[0], 'x1'),
    (2, 3, net_result[1], 'x2'),
    (3, 4, net_result[2], 'x3'),
]

for n1, n2, flow, name in internal_edges:
    x1, y1 = nodes[n1]
    x2, y2 = nodes[n2]
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2
    ax.annotate('', xy=(x2 - 0.35, y2), xytext=(x1 + 0.35, y1),
                arrowprops=dict(arrowstyle='->', color=internal_color,
                                lw=2.5, mutation_scale=20))
    ax.text(mx, my + 0.35, f'{name}={flow:.0f}', ha='center', va='center',
            fontsize=11, color=internal_color, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                      edgecolor=internal_color, alpha=0.9))

ax.annotate('', xy=(1, 1.65), xytext=(1, 2.2),
            arrowprops=dict(arrowstyle='->', color=external_color, lw=2, mutation_scale=15))
ax.text(1, 2.35, '300 in', ha='center', fontsize=10, color=external_color, fontweight='bold')

ax.annotate('', xy=(3, 2.2), xytext=(3, 1.65),
            arrowprops=dict(arrowstyle='->', color=external_color, lw=2, mutation_scale=15))
ax.text(3, 2.35, '100 out', ha='center', fontsize=10, color=external_color, fontweight='bold')

ax.annotate('', xy=(5, 2.2), xytext=(5, 1.65),
            arrowprops=dict(arrowstyle='->', color=external_color, lw=2, mutation_scale=15))
ax.text(5, 2.35, '150 out', ha='center', fontsize=10, color=external_color, fontweight='bold')

ax.annotate('', xy=(7, 0.0), xytext=(7, 0.65),
            arrowprops=dict(arrowstyle='->', color=external_color, lw=2, mutation_scale=15))
ax.text(7, -0.25, f'x4={net_result[3]:.0f} out', ha='center',
        fontsize=10, color=external_color, fontweight='bold')

plt.tight_layout()
plt.show()

---

## 6. Exercises

Test your understanding with the following problems. Write your solutions in the
code cells below.

### Exercise 1 (Easy)

Find the parabola $y = a + bx + cx^2$ passing through the three points
$(1, 5)$, $(2, 11)$, and $(3, 19)$.

**Steps:**
1. Build the $3 \times 3$ Vandermonde matrix using `build_vandermonde`.
2. Form the augmented matrix $[V \mid \mathbf{y}]$.
3. Solve using `solve()`.
4. Print the coefficients $a, b, c$ and write out the polynomial.
5. Verify by evaluating $p(x)$ at each data point.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium)

Consider a traffic network with 3 intersections and the following layout:

```
  400 (in)         200 (out)
     |                ^
     v                |
    [1] --- x1 ---> [2] --- x2 ---> [3]
                                     |
                                     v
                                   300 (out)
     ^                              
     |          x3                  
     +------------------------------+
```

Flows: 400 vehicles/hour enter at intersection 1. 200 leave at intersection 2.
300 leave at intersection 3. A return road $x_3$ goes from intersection 3
back to intersection 1.

**Steps:**
1. Write the flow conservation equation at each intersection.
2. Set up the augmented matrix and solve.
3. If the system is underdetermined, express the solution in terms of the
   free variable.
4. Determine the valid range of the free variable such that all flows $\geq 0$.
5. What is the maximum and minimum flow on the return road $x_3$?

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge)

Write a function `fit_polynomial(points, degree)` that:

1. Builds the Vandermonde matrix for the given degree.
2. Forms the augmented system and solves it.
3. Returns `(solution_type, coefficients_or_result)`.

**Test your function in these three regimes:**

- **Exact fit** ($\text{degree} = n - 1$ for $n$ points): Use the 4 data points
  from this notebook with degree 3. Should produce a unique solution.

- **Underdetermined** ($\text{degree} > n - 1$): Use the same 4 points with
  degree 5. What does the parametric solution mean? (There are infinitely many
  degree-5 polynomials through 4 points.)

- **Overdetermined** ($\text{degree} < n - 1$): Use the same 4 points with
  degree 1 (a line). What happens? Explain why the system is likely
  inconsistent (the 4 points are not collinear).

**Hint:** For the overdetermined case, `solve()` will return `"inconsistent"`
because there is no line through all four points. In practice, one uses
**least-squares** fitting (Chapter 5) to find the best-fit line.

In [ ]:
# YOUR CODE HERE


---

## Summary

| Concept | Key Takeaway |
|---|---|
| **Polynomial interpolation** | $n$ data points with distinct $x$-values uniquely determine a polynomial of degree $n-1$ |
| **Vandermonde matrix** | Row $i$ is $[1,\; x_i,\; x_i^2,\; \ldots]$; nonsingular when all $x_i$ are distinct |
| **Network flow** | Flow conservation (in = out) at each node gives a linear system |
| **Determined networks** | Same number of segments and independent equations yields a unique flow |
| **Underdetermined networks** | More segments than constraints produce free variables = adjustable flows |
| **Physical constraints** | Non-negativity of flows restricts the range of free parameters |
| **Vandermonde + `solve()`** | Build the matrix, augment with $\mathbf{y}$, solve --- a reusable pattern for curve fitting |
| **Verification strategy** | Always cross-check: evaluate at data points, compare with NumPy, check conservation equations |

**Next:** Notebook 2.1 --- Matrix Operations (addition, scalar multiplication, matrix multiplication).